# Rating-area polygons enriched with ZIP codes

Builds one polygon per ACA geographic rating area, per state, and lists the ZIP codes inside
it, so a sample ZIP can be drawn per rating area and sent to that state's Marketplace.

Sources
- **Definitions:** CMS state-specific geographic rating areas
  (`cms.gov/cciio/programs-and-initiatives/health-insurance-market-reforms/state-gra`, one page per
  state). Each page is a table of rating area × county, or × 3-digit ZIP prefix where the state
  defines areas that way (AK, MA, and Los Angeles County in CA). Pages are fetched once and cached
  under `data/reference/cms_gra/`.
- **Geometry:** TIGER 2025 counties (`data/tiger/tl_2025_us_county/`) dissolved per rating area; ZCTA polygons
  (`data/tiger/tl_2025_us_zcta520/`) for ZIP-prefix areas and for the ZIP lists. Connecticut uses the TIGER
  2021 counties (downloaded once) because CMS still defines its areas by the eight counties that
  TIGER replaced with planning regions in 2022.
- **Validation reference:** TIGER 2025 states (`data/tiger/tl_2025_us_state/`). The finished areas are checked
  against the state polygon, which is independent of the county file they were built from.

Output, one row per (state, rating area): `counties`, `county_fips`, `zip3`, three ZIP tiers by the
share of each ZCTA's area inside the polygon (`zips` ≥ 1%, `zips_majority` ≥ 50%,
`zips_no_overlap` ≥ 99%), counts, and the polygon. Saved to `data/reference/rating_areas.parquet`
(GeoParquet, list columns intact), a CSV with `|`-joined lists, and
`data/reference/rating_area_zips.json` (a compact export without geometry). Coverage results go to
`data/reference/rating_area_coverage.csv`.

Caveats: ZCTAs approximate USPS ZIPs, and PO-box ZIPs have no polygon. CMS pages contain a dozen
misspelled county names; these are fuzzy-matched and every fix is printed. New Jersey and Nebraska
publish different areas for the Individual and Small Group markets; the Individual table is used.
Territories are skipped.

In [ ]:
import difflib
import html
import re
import time
import warnings
from pathlib import Path

import addfips
import geopandas as gpd
import pandas as pd
import requests

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

COUNTY_SHAPEFILE = Path("data/tiger/tl_2025_us_county/tl_2025_us_county.shp")  # builds the rating-area polygons
STATE_SHAPEFILE = Path("data/tiger/tl_2025_us_state/tl_2025_us_state.shp")  # independent reference for validation
ZCTA_SHAPEFILE = Path("data/tiger/tl_2025_us_zcta520/tl_2025_us_zcta520.shp")
CMS_INDEX = "https://www.cms.gov/cciio/programs-and-initiatives/health-insurance-market-reforms/state-gra"
CMS_CACHE = Path("data/reference/cms_gra")
OUT_DIR = Path("data/reference/rating_areas")  # one GeoParquet per state
OUT_PARQUET = Path("data/reference/rating_areas.parquet")
OUT_CSV = Path("data/reference/rating_areas.csv")
USER_AGENT = "marketplace-research-scraper/0.1 (non-commercial research)"

# Connecticut replaced its 8 counties with 9 planning regions in TIGER 2022+, but CMS still defines
# its rating areas by the old counties. Those come from the 2021 TIGER county file, fetched once.
LEGACY_COUNTY_STATES = {"CT"}
LEGACY_COUNTY_URL = "https://www2.census.gov/geo/tiger/TIGER2021/COUNTY/tl_2021_us_county.zip"
LEGACY_COUNTY_ZIP = Path("data/reference/tiger_2021_county/tl_2021_us_county.zip")

EQUAL_AREA_CRS = "EPSG:5070"  # for area shares
OUTPUT_CRS = "EPSG:4326"
MIN_SHARE = 0.01  # a ZCTA is "in" an area when at least this share of its area lies inside
MAJORITY = 0.5  # ... "mostly in" at this share (what a ZIP-to-county table would assign)
FULLY_INSIDE = 0.99  # ... and "without overlap" at this share

TERRITORIES = {"AS", "GU", "MP", "PR", "VI"}
STATES = None  # None = every state on the CMS index except territories; or e.g. ["MD", "VA", "AK"]

fips = addfips.AddFIPS()

## Fetch and parse the CMS definitions

In [ ]:
def fetch_cms_pages(cache: Path = CMS_CACHE, delay: float = 1.0) -> dict[str, Path]:
    """Download the CMS index and each state's page once. Returns {state: cached html path}."""
    cache.mkdir(parents=True, exist_ok=True)
    headers = {"User-Agent": USER_AGENT}
    index = cache / "index.html"
    if not index.exists():
        r = requests.get(CMS_INDEX, headers=headers, timeout=30)
        r.raise_for_status()
        index.write_text(r.content.decode("utf-8", "replace"))
        time.sleep(delay)
    # Nebraska's links carry a #fragment per market, hence the optional tail before the closing quote.
    links = re.findall(r'href="(/cciio/programs-and-initiatives/health-insurance-market-reforms/([a-z]{2})-gra)(?:#[^"]*)?"', index.read_text(), re.I)
    pages = {}
    for href, st in sorted(set(links), key=lambda kv: kv[1]):
        state = st.upper()
        path = cache / f"{st.lower()}.html"
        if not path.exists():
            r = requests.get("https://www.cms.gov" + href, headers=headers, timeout=30)
            r.raise_for_status()
            path.write_text(r.content.decode("utf-8", "replace"))
            time.sleep(delay)
        pages[state] = path
    return pages


_TAG = re.compile(r"<.*?>", re.S)


def _clean(fragment: str) -> str:
    text = html.unescape(_TAG.sub("", fragment))
    text = re.sub(r"[\xa0Â]", " ", text)  # non-breaking spaces, including the mojibake form
    return re.sub(r"\s+", " ", text).strip()


def parse_cms_page(path: Path) -> pd.DataFrame:
    """Rows of the state's Individual-market table: rating_area, county, zip3, note."""
    text = path.read_text()
    tables = [(_clean(text[max(0, m.start() - 600) : m.start()]).lower(), m.group(0)) for m in re.finditer(r"<table.*?</table>", text, re.S)]
    if not tables:
        raise ValueError(f"{path}: no rating-area table found")
    # NJ and NE publish Individual and Small Group tables; take the Individual one. MI repeats one table twice.
    table = next((t for context, t in tables if "individual" in context), tables[0][1])
    rows = []
    for tr in re.findall(r"<tr.*?</tr>", table, re.S):
        cells = [_clean(c) for c in re.findall(r"<t[hd][^>]*>(.*?)</t[hd]>", tr, re.S)]
        if not cells or cells[0].lower().startswith("rating area id"):
            continue
        area = re.search(r"(\d+)", cells[0])
        if not area:
            continue
        county = cells[1] if len(cells) > 1 else ""
        zip_cell = cells[2] if len(cells) > 2 else ""
        zip3 = re.match(r"(\d{3})", zip_cell)
        note = zip_cell[zip3.end() :].strip(" ()") if zip3 else ""
        if not county and not zip3:
            continue
        rows.append({"rating_area": int(area.group(1)), "county": county, "zip3": zip3.group(1) if zip3 else "", "note": note})
    return pd.DataFrame(rows, columns=["rating_area", "county", "zip3", "note"]).drop_duplicates().reset_index(drop=True)


pages = fetch_cms_pages()
states = [s for s in (STATES or pages) if s not in TERRITORIES]
print(len(pages), "pages cached;", len(states), "states selected")
parse_cms_page(pages["MD"]).head(8)

## Resolve county names to FIPS

`addfips` first, then an exact normalized match on the TIGER names, then a fuzzy match for the
misspellings. Fuzzy fixes and anything still unmatched are returned as notes and printed per state.

In [ ]:
COUNTY_ALIASES = {"la": "Los Angeles"}  # abbreviations used in CMS notes, e.g. "only within LA County"
_SUFFIX = re.compile(r"\s+(county|parish|borough|census area|municipality|city and borough)$", re.I)
_LSAD_SUFFIXES = ("county", "parish", "borough", "censusarea", "municipality", "cityandborough", "city")


def _norm(name: str) -> str:
    return re.sub(r"[^a-z0-9]", "", name.lower().replace("saint ", "st "))


def county_source(state: str) -> str:
    """Path pyogrio can read for the county layer that matches how CMS names this state's counties."""
    if state not in LEGACY_COUNTY_STATES:
        return str(COUNTY_SHAPEFILE)
    if not LEGACY_COUNTY_ZIP.exists():
        LEGACY_COUNTY_ZIP.parent.mkdir(parents=True, exist_ok=True)
        with requests.get(LEGACY_COUNTY_URL, headers={"User-Agent": USER_AGENT}, timeout=120, stream=True) as r:
            r.raise_for_status()
            with LEGACY_COUNTY_ZIP.open("wb") as f:
                for chunk in r.iter_content(1 << 20):
                    f.write(chunk)
    return f"/vsizip/{LEGACY_COUNTY_ZIP.resolve()}/{LEGACY_COUNTY_ZIP.stem}.shp"


def load_counties(state: str) -> gpd.GeoDataFrame:
    state_fips = fips.get_state_fips(state)
    gdf = gpd.read_file(county_source(state), where=f"STATEFP = '{state_fips}'", columns=["STATEFP", "GEOID", "NAME", "NAMELSAD"], engine="pyogrio")
    if gdf.empty:
        raise ValueError(f"no counties for {state} (FIPS {state_fips}) in {county_source(state)}")
    return gdf.to_crs(EQUAL_AREA_CRS)


def resolve_county(name: str, state: str, tiger: gpd.GeoDataFrame) -> tuple[str | None, str | None]:
    """County name as CMS spells it -> (GEOID, note). The note says how it was matched if not exactly.

    Order matters: a bare name such as "Fairfax" must match "Fairfax County" before anything
    else, because Virginia and Missouri also have an independent "Fairfax city" / "St. Louis city"
    that generic lookups prefer. CMS writes the cities as "Fairfax City", which matches NAMELSAD.
    """
    name = COUNTY_ALIASES.get(name.strip().lower(), name.strip())
    by_lsad = {_norm(n): g for n, g in zip(tiger["NAMELSAD"], tiger["GEOID"])}
    key = _norm(name)
    if key in by_lsad:
        return by_lsad[key], None
    for suffix in _LSAD_SUFFIXES:
        if key + suffix in by_lsad:
            return by_lsad[key + suffix], None
    code = fips.get_county_fips(name, state=state)
    if code and code in by_lsad.values():
        return code, None
    by_name = {_norm(n): g for n, g in zip(tiger["NAME"], tiger["GEOID"])}
    bare = _norm(_SUFFIX.sub("", name))
    close = difflib.get_close_matches(bare, list(by_name), n=1, cutoff=0.8)
    if close:
        code = by_name[close[0]]
        matched = tiger.loc[tiger["GEOID"] == code, "NAMELSAD"].iloc[0]
        return code, f"{name!r} -> {matched!r} (fuzzy)"
    return None, f"{name!r} UNMATCHED"


def resolve_counties(rows: pd.DataFrame, state: str, tiger: gpd.GeoDataFrame) -> tuple[pd.DataFrame, list[str]]:
    codes, notes = [], []
    for county in rows["county"]:
        if not county:
            codes.append("")
            continue
        code, note = resolve_county(county, state, tiger)
        codes.append(code or "")
        if note:
            notes.append(note)
    return rows.assign(county_fips=codes), notes

## Build polygons and ZIP lists

County rows dissolve county polygons. ZIP-prefix rows dissolve the ZCTAs with that prefix, clipped
to the state (and to a county when the CMS note restricts them, as for Los Angeles). ZIP lists come
from the share of each ZCTA's area that falls inside the finished polygon.

In [ ]:
def load_zctas(where: str | None = None, bbox=None) -> gpd.GeoDataFrame:
    gdf = gpd.read_file(ZCTA_SHAPEFILE, where=where, bbox=bbox, columns=["ZCTA5CE20"], engine="pyogrio")
    return gdf.rename(columns={"ZCTA5CE20": "zcta"}).to_crs(EQUAL_AREA_CRS)


def restricted_county(note: str, state: str, tiger: gpd.GeoDataFrame) -> str | None:
    """GEOID named by a CMS note such as 'only within LA County', else None."""
    match = re.search(r"only within (.+?) county", note or "", re.I)
    if not match:
        return None
    code, _ = resolve_county(match.group(1), state, tiger)
    return code


def build_area_polygons(rows: pd.DataFrame, state: str, tiger: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """One dissolved polygon per rating area from county rows and/or ZIP-prefix rows."""
    parts = []
    county_rows = rows[rows["county_fips"] != ""]
    if len(county_rows):
        merged = tiger.merge(county_rows[["rating_area", "county_fips"]], left_on="GEOID", right_on="county_fips")
        parts.append(merged[["rating_area", "geometry"]])
    zip_rows = rows[rows["zip3"] != ""]
    if len(zip_rows):
        where = " OR ".join(f"ZCTA5CE20 LIKE '{p}%'" for p in sorted(set(zip_rows["zip3"])))
        zctas = load_zctas(where=where)
        zctas["zip3"] = zctas["zcta"].str[:3]
        zctas = zctas.merge(zip_rows[["rating_area", "zip3", "note"]], on="zip3")
        state_geom = tiger.geometry.union_all()
        zctas["geometry"] = zctas.geometry.intersection(state_geom)
        for note in zctas["note"].unique():
            code = restricted_county(note, state, tiger)
            if code:
                county_geom = tiger.loc[tiger["GEOID"] == code].geometry.union_all()
                mask = zctas["note"] == note
                zctas.loc[mask, "geometry"] = zctas.loc[mask].geometry.intersection(county_geom)
        zctas = zctas[zctas.geometry.area > 0]
        parts.append(zctas[["rating_area", "geometry"]])
    if not parts:
        raise ValueError(f"{state}: no county or ZIP rows resolved")
    areas = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), crs=EQUAL_AREA_CRS).dissolve(by="rating_area").reset_index()
    return areas[["rating_area", "geometry"]]


ZIP_TIERS = {"zips": MIN_SHARE, "zips_majority": MAJORITY, "zips_no_overlap": FULLY_INSIDE}


def enrich_zips(areas: gpd.GeoDataFrame, tiger: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Add the ZIP tiers: `zips` (>= 1% of the ZCTA inside), `zips_majority` (>= 50%), `zips_no_overlap` (>= 99%)."""
    zctas = load_zctas(bbox=tuple(tiger.to_crs("EPSG:4269").total_bounds))
    zctas["zcta_area"] = zctas.geometry.area
    pairs = gpd.sjoin(zctas, areas[["rating_area", "geometry"]], how="inner", predicate="intersects")
    area_geoms = areas.set_index("rating_area").geometry
    inside = pairs.geometry.intersection(gpd.GeoSeries(area_geoms.loc[pairs["rating_area"]].values, index=pairs.index, crs=EQUAL_AREA_CRS))
    pairs["share"] = inside.area / pairs["zcta_area"]
    out = areas.copy()
    for column, threshold in ZIP_TIERS.items():
        tier = pairs[pairs["share"] >= threshold].groupby("rating_area")["zcta"].agg(lambda s: sorted(s)).rename(column)
        out = out.merge(tier, left_on="rating_area", right_index=True, how="left")
        out[column] = out[column].apply(lambda v: v if isinstance(v, list) else [])
    return out


def build_state(state: str) -> tuple[gpd.GeoDataFrame, dict]:
    rows = parse_cms_page(pages[state])
    tiger = load_counties(state)
    rows, notes = resolve_counties(rows, state, tiger)
    areas = enrich_zips(build_area_polygons(rows, state, tiger), tiger)

    names = tiger.set_index("GEOID")["NAMELSAD"]
    county_rows = rows[rows["county_fips"] != ""]
    by_area = county_rows.groupby("rating_area")["county_fips"].agg(lambda s: sorted(set(s)))
    zip3_by_area = rows[rows["zip3"] != ""].groupby("rating_area")["zip3"].agg(lambda s: sorted(set(s)))
    areas["county_fips"] = areas["rating_area"].map(by_area).apply(lambda v: v if isinstance(v, list) else [])
    areas["counties"] = areas["county_fips"].apply(lambda codes: [names[c] for c in codes])
    areas["zip3"] = areas["rating_area"].map(zip3_by_area).apply(lambda v: v if isinstance(v, list) else [])
    areas.insert(0, "state", state)
    areas["n_counties"] = areas["counties"].str.len()
    for column in ZIP_TIERS:
        areas[f"n_{column}"] = areas[column].str.len()
    areas = areas.to_crs(OUTPUT_CRS)

    covered = set(county_rows["county_fips"]) | {restricted_county(n, state, tiger) for n in rows["note"].unique()}
    report = {
        "state": state,
        "areas": len(areas),
        "county_rows": len(county_rows),
        "zip3_rows": int((rows["zip3"] != "").sum()),
        "tiger_counties": len(tiger),
        "counties_missing": sorted(names[c] for c in set(tiger["GEOID"]) - covered) if len(county_rows) else "(defined by ZIP prefix)",
        "counties_in_two_areas": sorted(names[c] for c, n in county_rows.groupby("county_fips")["rating_area"].nunique().items() if n > 1),
        "notes": notes,
    }
    return areas, report


md_areas, md_report = build_state("MD")
print(md_report)
md_areas[["state", "rating_area", "n_counties", "n_zips", "n_zips_majority", "n_zips_no_overlap", "counties"]]

## Validate coverage

The question all three tests answer is: *if someone anywhere in the state types their ZIP into
the Marketplace site, do we have a rating area for them?* The reference for "the state" is the
TIGER state polygon in `data/tiger/tl_2025_us_state/`, which is independent of the county file used to build the
areas. Tests 1 and 2 check the geometry against itself; Test 3 checks it against an independent
list of real ZIP codes.

### Test 1: area coverage

**What is compared.** The union of all of a state's rating-area polygons against the state's own
polygon from the state shapefile.

**What is measured.**
- `area_coverage_pct`: the share of the state polygon that lies inside some rating area.
- `gap_km2`: state area inside no rating area. The five largest gaps are named by the county
  that contains them (`largest_gaps`), so a missing county is identified by name.
- `outside_state_km2`: rating-area polygon that falls outside the state polygon. This should be
  near zero; a large value means the areas were built from the wrong geometry.
- `overlap_km2`: area claimed by two rating areas at once. This should be zero; a county listed
  under two areas on the CMS page would show up here.

**How to read it.** In a state that defines rating areas by county, the areas are whole counties
and counties tile the state, so coverage must be 100%. Anything less means the CMS page left a
county out or a county name failed to match. In a state that defines areas by 3-digit ZIP prefix
(Alaska, Massachusetts, and Los Angeles County in California) the polygons are built from ZCTAs,
and ZCTAs do not exist over open water or unpopulated land such as national forest. Those states
therefore show honest gaps (Los Angeles: about 2,000 km² of ocean and mountains) that are **not**
errors. That is why this test is only enforced for county-defined states, and why Test 2 exists.

### Test 2: ZCTA coverage

Area alone cannot tell us whether a prefix-defined state is complete, because its gaps are
expected. So this test counts ZIP-code polygons instead of square kilometres.

**Step 1: find the ZCTAs that belong to the state.** Take every ZCTA polygon and compute the share
of its area inside the state polygon. A ZCTA with at least 50% inside counts as in-state
(`zctas_in_state`). This drops ZCTAs from neighbouring states that only poke across the border.

**Step 2: look each one up in the rating-area ZIP lists.** Each rating area carries three lists,
by how much of the ZCTA's area falls inside that area's polygon: `zips` (at least 1%),
`zips_majority` (at least 50%), and `zips_no_overlap` (at least 99%).

**Step 3: classify every in-state ZCTA into exactly one of three outcomes.**

| Outcome | Definition | Meaning |
|---|---|---|
| Assigned | In some area's `zips_majority` list | We know which rating area prices this ZIP. |
| No majority (`zctas_no_majority`) | In at least one area's `zips` list, but in no `zips_majority` list | The ZIP is split across areas, for example 40% / 35% / 25%. It is covered, but geometry cannot say which area the site will rate it in, so it is never sampled. |
| Uncovered (`zctas_uncovered`) | In no area's list at all | A ZIP polygon in the state that no rating area reaches. **This is the failure.** |

**What an uncovered ZCTA would mean.** In a prefix-defined state, an area is literally "every ZCTA
whose ZIP starts with 906, 907, 908, …". If the CMS table omitted a prefix, or a row was parsed
wrongly, every ZIP with that prefix would land in no area. Test 1 could not see this, because
those states never reach 100% area coverage anyway. Test 2 lists the uncovered ZCTAs
(`uncovered_zctas`) and counts them by 3-digit prefix (`uncovered_prefixes`), so a missing prefix
shows up as, say, `{"901": 14}`.

**Worked examples from the current build.**
- *Los Angeles County.* 295 ZCTAs lie mostly inside the county. All 295 are in the majority list
  of rating area 15 or 16, none is uncovered. So the blank patches on the map are land and water
  with no ZCTA, not missing prefixes.
- *Virginia.* 903 in-state ZCTAs: 900 assigned, 3 with no majority (23149, 23888, 24553, each
  straddling three rating areas), 0 uncovered.

**The blind spot of this test.** Its population (ZCTAs) comes from the same shapefile that built
the ZIP lists. A real ZIP with no polygon is invisible to it. Test 3 closes that gap.

### Test 3: ZIP-library coverage

**What is compared.** Every ZIP code the `zipcodes` library lists for the state (a USPS-derived
list with no geometry, and the list the scenario generator draws from) against that same state's
rating-area ZIP lists. Because the library was not used to build the lists, this is an
independent check, and it reflects what a person can actually type into a site.

**Step 1: take the library's ZIPs for the state**, and note each one's type. "Standard" here
means type `STANDARD` and active: an ordinary delivery ZIP, which is what the scrapers quote.
"Other" is everything else: PO-box ZIPs, single-organisation (`UNIQUE`) ZIPs, and retired ZIPs.

**Step 2: place each ZIP in the tightest tier that contains it**, looking only at this state's
rating areas.

| Tier (column suffix) | Definition | Meaning |
|---|---|---|
| `no_overlap` | In some area's `zips_no_overlap` list | At least 99% of the ZIP's polygon is inside one rating area. Safe to sample. |
| `majority` | Not above, but in some `zips_majority` list | 50–99% inside one area. Placed, but it touches a boundary. |
| `partial` | Not above, but in some `zips` list | Touches one or more areas with no majority. Covered, ambiguous. |
| `no_polygon` | In none of the lists, and the ZIP has no ZCTA polygon at all | Cannot be placed by geometry. Expected for PO-box and single-organisation ZIPs; also a few small standard ZIPs the Census did not turn into ZCTAs. Not a failure of the rating areas. |
| `unplaced` | Has a ZCTA polygon, yet is in none of this state's lists | **This is the failure.** Either the polygon sits in a different state from the one the library names, or the lists are missing something. |

Columns are named `lib_standard_<tier>` and `lib_other_<tier>`; `lib_unplaced_zips` lists the
failures.

**Reverse direction.** `majority_zips_library_files_elsewhere` lists ZCTAs that geometry places
mostly in this state but that the library files under a different state. These are border ZIPs
whose post office is across the state line (for example 97635, an Oregon ZIP whose delivery
area is mostly in California). They are informational: the sampler already excludes them,
because it only draws ZIPs the library lists for the state.

**Result on the current build, all 51 jurisdictions.** Of 29,673 standard ZIPs: 25,934 (87.4%)
`no_overlap`, 3,394 (11.4%) `majority`, 44 (0.15%) `partial`, 301 (1.0%) `no_polygon`,
0 `unplaced`. Of the 12,076 other ZIPs, 4,270 have a polygon and are placed, 7,806 have no
polygon, 0 are unplaced. Eleven ZCTAs are filed by the library under a neighbouring state.

### Status

`status` is `OK` when no ZCTA is uncovered (Test 2), no library ZIP is unplaced (Test 3), and,
for county-defined states, area coverage is at least 99.99% (Test 1). No-majority ZCTAs,
`partial` ZIPs and `no_polygon` ZIPs do not fail the test; they are reported so you know which
ZIPs to avoid or handle separately.

In [ ]:
from collections import Counter

import pyogrio
import zipcodes

_ALL_ZCTAS: set[str] | None = None
LIBRARY_TIERS = ("no_overlap", "majority", "partial", "no_polygon", "unplaced")


def load_state(state: str) -> gpd.GeoDataFrame:
    """The state's polygon from the TIGER state file: the reference the areas are validated against."""
    gdf = gpd.read_file(STATE_SHAPEFILE, where=f"STUSPS = '{state}'", columns=["STUSPS", "NAME", "ALAND", "AWATER"], engine="pyogrio")
    if gdf.empty:
        raise ValueError(f"{state} not found in {STATE_SHAPEFILE}")
    return gdf.to_crs(EQUAL_AREA_CRS)


def all_zctas() -> set[str]:
    """Every ZCTA code in the shapefile (attribute table only, read once)."""
    global _ALL_ZCTAS
    if _ALL_ZCTAS is None:
        _ALL_ZCTAS = set(pyogrio.read_dataframe(ZCTA_SHAPEFILE, read_geometry=False, columns=["ZCTA5CE20"])["ZCTA5CE20"])
    return _ALL_ZCTAS


def library_coverage(state: str, areas: gpd.GeoDataFrame) -> dict:
    """Test 3: place every ZIP the `zipcodes` library lists for the state in the tightest tier holding it."""
    no_overlap = {z for zips in areas["zips_no_overlap"] for z in zips}
    majority = {z for zips in areas["zips_majority"] for z in zips}
    touched = {z for zips in areas["zips"] for z in zips}
    library = zipcodes.filter_by(state=state)
    counts, unplaced = Counter(), []
    for entry in library:
        code = entry["zip_code"]
        kind = "standard" if entry["zip_code_type"] == "STANDARD" and entry["active"] else "other"
        if code in no_overlap:
            tier = "no_overlap"
        elif code in majority:
            tier = "majority"
        elif code in touched:
            tier = "partial"
        elif code not in all_zctas():
            tier = "no_polygon"
        else:
            tier = "unplaced"
            unplaced.append(code)
        counts[(kind, tier)] += 1
    out = {f"lib_{kind}_{tier}": counts[(kind, tier)] for kind in ("standard", "other") for tier in LIBRARY_TIERS}
    out["lib_standard_total"] = sum(counts[("standard", tier)] for tier in LIBRARY_TIERS)
    out["lib_unplaced_zips"] = sorted(unplaced)[:50]
    # Reverse direction: ZCTAs we place mostly in this state that the library files under another state.
    library_codes = {entry["zip_code"] for entry in library}
    out["majority_zips_library_files_elsewhere"] = sorted(majority - library_codes)[:50]
    return out


def validate_coverage(state: str, areas: gpd.GeoDataFrame) -> dict:
    """Tests 1 (area), 2 (ZCTA) and 3 (ZIP library) for one state; see the section above."""
    polygons = areas.to_crs(EQUAL_AREA_CRS)
    state_geom = load_state(state).geometry.union_all()
    covered = polygons.geometry.union_all()

    # --- Test 1: area ---
    gap = state_geom.difference(covered)
    outside = covered.difference(state_geom)
    area_pct = covered.intersection(state_geom).area / state_geom.area * 100
    overlap_km2 = (polygons.geometry.area.sum() - covered.area) / 1e6
    has_counties = polygons["n_counties"].sum() > 0
    has_zip3 = polygons["zip3"].str.len().sum() > 0
    defined_by = "county" if has_counties and not has_zip3 else "zip3" if has_zip3 and not has_counties else "mixed"

    # Name the five largest gaps (>= 1 km2) by the county containing a point inside each.
    gap_notes = []
    if not gap.is_empty:
        pieces = gpd.GeoDataFrame(geometry=gpd.GeoSeries([gap], crs=EQUAL_AREA_CRS)).explode(ignore_index=True)
        pieces["km2"] = pieces.geometry.area / 1e6
        pieces = pieces[pieces["km2"] >= 1].sort_values("km2", ascending=False).head(5)
        if len(pieces):
            counties = load_counties(state)
            points = pieces.set_geometry(pieces.representative_point())
            located = gpd.sjoin(points, counties[["NAMELSAD", "geometry"]], how="left", predicate="within")
            gap_notes = [f"{row.NAMELSAD if isinstance(row.NAMELSAD, str) else '?'} {row.km2:,.0f} km2" for row in located.itertuples()]

    # --- Test 2: ZCTAs ---
    # Step 1: ZCTAs with at least half their area inside the state polygon.
    minx, miny, maxx, maxy = gpd.GeoSeries([state_geom], crs=EQUAL_AREA_CRS).to_crs("EPSG:4269").total_bounds
    zctas = load_zctas(bbox=(minx, miny, maxx, maxy))
    share_in_state = zctas.geometry.intersection(state_geom).area / zctas.geometry.area
    in_state = set(zctas.loc[share_in_state >= MAJORITY, "zcta"])
    # Step 2: what the rating-area lists say about them.
    touched = {z for zips in polygons["zips"] for z in zips}  # >= 1% inside some area
    assigned = {z for zips in polygons["zips_majority"] for z in zips}  # >= 50% inside some area
    # Step 3: classify.
    uncovered = sorted(in_state - touched)  # in no area's list: the failure
    no_majority = sorted((in_state & touched) - assigned)  # split across areas: covered, ambiguous

    # --- Test 3: ZIP library ---
    library = library_coverage(state, polygons)
    library_unplaced = library["lib_standard_unplaced"] + library["lib_other_unplaced"]

    area_ok = area_pct >= 99.99 or defined_by != "county"
    return {
        "state": state,
        "status": "OK" if area_ok and not uncovered and not library_unplaced else "CHECK",
        "defined_by": defined_by,
        "area_coverage_pct": round(area_pct, 2),
        "gap_km2": round(gap.area / 1e6, 1),
        "outside_state_km2": round(outside.area / 1e6, 1),
        "overlap_km2": round(overlap_km2, 1),
        "largest_gaps": "; ".join(gap_notes),
        "zctas_in_state": len(in_state),
        "zctas_assigned": len(in_state & assigned),
        "zctas_no_majority": len(no_majority),
        "zctas_uncovered": len(uncovered),
        "uncovered_prefixes": dict(Counter(z[:3] for z in uncovered)),
        "uncovered_zctas": uncovered[:50],
        "no_majority_zctas": no_majority[:50],
        **library,
    }


ca_path = OUT_DIR / "CA.parquet"
ca_areas = gpd.read_parquet(ca_path) if ca_path.exists() else build_state("CA")[0]
pd.DataFrame([validate_coverage("MD", md_areas), validate_coverage("CA", ca_areas)]).set_index("state").T

## Build every selected state

Each state is cached as `data/reference/rating_areas/<ST>.parquet`; delete a file to rebuild it.
The validation table flags counties the CMS page leaves out or lists twice, and prints every fuzzy
name fix. A county missing from a county-defined state means the CMS page and TIGER disagree and
the polygon has a hole there.

In [ ]:
import json
from datetime import datetime, timezone

OUT_JSON = Path("data/reference/rating_area_zips.json")  # compact export: no geometry
OUT_COVERAGE = Path("data/reference/rating_area_coverage.csv")

OUT_DIR.mkdir(parents=True, exist_ok=True)
frames, coverage_rows, failures = [], [], {}
for state in states:
    cached = OUT_DIR / f"{state}.parquet"
    coverage_cached = OUT_DIR / f"{state}.coverage.json"
    try:
        if cached.exists():
            areas = gpd.read_parquet(cached)
        else:
            t = time.time()
            areas, report = build_state(state)
            areas.to_parquet(cached)
            flags = []
            if report["counties_missing"] and isinstance(report["counties_missing"], list):
                flags.append(f"missing {report['counties_missing']}")
            if report["counties_in_two_areas"]:
                flags.append(f"in two areas {report['counties_in_two_areas']}")
            flags += report["notes"]
            print(f"{state}: {report['areas']} areas, {report['county_rows']} county rows, {report['zip3_rows']} zip3 rows, {time.time() - t:.1f}s" + ("  |  " + "; ".join(flags) if flags else ""))
        frames.append(areas)
        if coverage_cached.exists():
            coverage = json.loads(coverage_cached.read_text())
        else:
            coverage = validate_coverage(state, areas)
            coverage_cached.write_text(json.dumps(coverage))
        coverage_rows.append(coverage)
    except Exception as err:
        failures[state] = str(err)
        print(f"{state}: FAILED {err}")

rating_areas = gpd.GeoDataFrame(pd.concat(frames, ignore_index=True), crs=OUTPUT_CRS)
rating_areas.to_parquet(OUT_PARQUET)
flat = rating_areas.drop(columns="geometry").copy()
for col in ("counties", "county_fips", "zip3", *ZIP_TIERS):
    flat[col] = flat[col].apply("|".join)
flat.to_csv(OUT_CSV, index=False)

# Compact export: no geometry, lists intact.
export = {
    "generated_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "source": CMS_INDEX,
    "tiers": {"zips": MIN_SHARE, "zips_majority": MAJORITY, "zips_no_overlap": FULLY_INSIDE},
    "states": {
        state: {
            str(row.rating_area): {col: list(getattr(row, col)) for col in ("counties", "county_fips", "zip3", *ZIP_TIERS)}
            for row in group.sort_values("rating_area").itertuples()
        }
        for state, group in rating_areas.groupby("state")
    },
}
OUT_JSON.write_text(json.dumps(export))

coverage = pd.DataFrame(coverage_rows).set_index("state")
coverage.to_csv(OUT_COVERAGE)
print(f"\n{len(rating_areas)} rating areas across {rating_areas['state'].nunique()} states -> {OUT_PARQUET}, {OUT_CSV}, {OUT_JSON}")
print(f"coverage -> {OUT_COVERAGE}: {int((coverage['status'] == 'OK').sum())} OK, {int((coverage['status'] != 'OK').sum())} to check")
if failures:
    print("failed:", failures)
coverage[coverage["status"] != "OK"]

## Inspect a state on the map

Pick a state to overlay three layers. Each can be switched on and off with the checkboxes in the
top-right layer control.

- **Rating areas**: the resolved polygons, one colour per area. Hover for the area number, the
  counties or ZIP prefixes that define it, and its ZCTA counts.
- **Counties**: dashed outlines from the county file the areas were built from. Hover for the
  county name and the rating area CMS assigns it. Where areas are defined by ZIP prefix
  (Alaska, Massachusetts, Los Angeles County) a county has no single area and says so.
- **ZCTAs**: every ZCTA with at least 1% of its area inside one of the state's rating areas,
  coloured by how cleanly it sits in one area: green when at least 99% is inside a single area,
  orange when 50–99% is, red when no area holds half. Hover for the exact split, for example
  `area 7: 45%, area 9: 30%, area 12: 25%`.

Hover reports the topmost visible layer under the cursor, so switch ZCTAs off to read counties,
and both off to read rating areas. Geometry is simplified (about 200 m) to keep the map light, so
shapes are slightly coarser than the stored polygons; the shares are computed on full-resolution
geometry.

In [ ]:
import math

import ipywidgets as widgets
import numpy as np
from ipyleaflet import FullScreenControl, GeoJSON, LayersControl, LegendControl, Map, WidgetControl, basemaps

AREA_COLORS = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
    "#aec7e8", "#ffbb78", "#98df8a", "#ff9896", "#c5b0d5", "#c49c94", "#f7b6d2", "#c7c7c7", "#dbdb8d", "#9edae5",
]  # cycled when a state has more than 20 areas
TIER_COLORS = {"no_overlap": "#1a9850", "majority": "#f39c12", "partial": "#d73027"}
TIER_LABELS = {"no_overlap": "≥ 99% in one area", "majority": "50–99% in one area", "partial": "no area holds half"}


def zoom_for_bounds(minx, miny, maxx, maxy, width_px=1000, height_px=700) -> int:
    """Largest Leaflet zoom at which the bounds fit in a viewport of the given size."""
    lon_span = max(maxx - minx, 1e-6)
    lat_span = max(maxy - miny, 1e-6) * 1.4  # rough Mercator stretch at mid latitudes
    zoom = min(math.log2(width_px * 360 / (256 * lon_span)), math.log2(height_px * 360 / (256 * lat_span)))
    return int(max(2, min(12, math.floor(zoom))))


def state_layers(state: str, simplify_deg: float = 0.002) -> tuple[gpd.GeoDataFrame, gpd.GeoDataFrame, gpd.GeoDataFrame]:
    """(rating areas, counties, ZCTAs) for one state, in WGS84 and simplified for the browser."""
    areas = rating_areas[rating_areas["state"] == state].sort_values("rating_area").copy()
    if areas.empty:
        raise ValueError(f"no rating areas built for {state}")
    areas_ea = areas.to_crs(EQUAL_AREA_CRS)

    counties = load_counties(state)
    area_of_county = {code: row.rating_area for row in areas.itertuples() for code in row.county_fips}
    counties["rating_area"] = counties["GEOID"].map(area_of_county)

    # ZCTAs in the state's lists, with the share of each that falls in each area (full-resolution geometry).
    wanted = {z for zips in areas["zips"] for z in zips}
    zctas = load_zctas(bbox=tuple(counties.to_crs("EPSG:4269").total_bounds))
    zctas = zctas[zctas["zcta"].isin(wanted)].reset_index(drop=True)
    pairs = gpd.sjoin(zctas, areas_ea[["rating_area", "geometry"]], how="inner", predicate="intersects").reset_index(drop=True)
    area_geoms = areas_ea.set_index("rating_area").geometry
    inside = pairs.geometry.intersection(gpd.GeoSeries(area_geoms.loc[pairs["rating_area"]].values, index=pairs.index, crs=EQUAL_AREA_CRS))
    pairs["share"] = inside.area / pairs.geometry.area
    pairs = pairs[pairs["share"] >= MIN_SHARE].sort_values("share", ascending=False)
    top = pairs.groupby("zcta").first()
    split = pairs.groupby("zcta")[["rating_area", "share"]].apply(lambda g: ", ".join(f"area {a}: {s:.0%}" for a, s in zip(g["rating_area"], g["share"])))
    zctas["top_area"] = zctas["zcta"].map(top["rating_area"])
    zctas["top_share"] = zctas["zcta"].map(top["share"])
    zctas["tier"] = np.select([zctas["top_share"] >= FULLY_INSIDE, zctas["top_share"] >= MAJORITY], ["no_overlap", "majority"], default="partial")
    zctas["shares"] = zctas["zcta"].map(split)

    def simplified(gdf: gpd.GeoDataFrame, tolerance: float) -> gpd.GeoDataFrame:
        gdf = gdf.to_crs(OUTPUT_CRS)
        gdf["geometry"] = gdf.geometry.simplify(tolerance, preserve_topology=True)
        return gdf

    areas["counties_text"] = areas["counties"].apply(lambda c: ", ".join(c[:6]) + (f" … (+{len(c) - 6} more)" if len(c) > 6 else ""))
    areas["zip3_text"] = areas["zip3"].apply(", ".join)
    area_cols = ["rating_area", "n_counties", "n_zips", "n_zips_majority", "n_zips_no_overlap", "counties_text", "zip3_text", "geometry"]
    return (
        simplified(areas[area_cols], simplify_deg),
        simplified(counties[["GEOID", "NAMELSAD", "rating_area", "geometry"]], simplify_deg),
        simplified(zctas[["zcta", "tier", "top_area", "top_share", "shares", "geometry"]], simplify_deg / 2),
    )


def build_state_map(state: str, simplify_deg: float = 0.002, height: str = "700px") -> Map:
    """Rating areas, counties and ZCTAs for one state as three toggleable layers."""
    areas, counties, zctas = state_layers(state, simplify_deg)
    color_of = {area: AREA_COLORS[i % len(AREA_COLORS)] for i, area in enumerate(areas["rating_area"])}
    info = widgets.HTML(value="<i>hover a shape</i>")

    def describe_area(p: dict) -> str:
        defined = f"{p['n_counties']} counties: {p['counties_text']}" if p["n_counties"] else f"ZIP prefixes: {p['zip3_text']}"
        return (
            f"<b>{state} rating area {p['rating_area']}</b><br>{defined}<br>"
            f"ZCTAs: {p['n_zips']} touching, {p['n_zips_majority']} majority, {p['n_zips_no_overlap']} fully inside"
        )

    def describe_county(p: dict) -> str:
        area = f"rating area {int(p['rating_area'])}" if p["rating_area"] is not None else "no single area: defined by ZIP prefix here"
        return f"<b>{p['NAMELSAD']}</b> (FIPS {p['GEOID']})<br>{area}"

    def describe_zcta(p: dict) -> str:
        return f"<b>ZCTA {p['zcta']}</b><br>{TIER_LABELS[p['tier']]}<br>{p['shares']}"

    def layer(gdf: gpd.GeoDataFrame, name: str, describe, style: dict, hover_style: dict, style_callback=None) -> GeoJSON:
        kwargs = {"style_callback": style_callback} if style_callback else {}
        geo = GeoJSON(data=json.loads(gdf.to_json()), name=name, style=style, hover_style=hover_style, **kwargs)
        geo.on_hover(lambda **event: setattr(info, "value", describe(event["feature"]["properties"])))
        return geo

    area_layer = layer(
        areas, f"Rating areas ({len(areas)})", describe_area,
        style={"weight": 2, "fillOpacity": 0.35}, hover_style={"fillOpacity": 0.6},
        style_callback=lambda f: {"color": color_of[f["properties"]["rating_area"]], "fillColor": color_of[f["properties"]["rating_area"]]},
    )
    county_layer = layer(
        counties, f"Counties ({len(counties)})", describe_county,
        style={"color": "#222222", "weight": 1.2, "dashArray": "4 3", "fillOpacity": 0.0}, hover_style={"weight": 3, "fillOpacity": 0.1},
    )
    zcta_layer = layer(
        zctas, f"ZCTAs ({len(zctas)})", describe_zcta,
        style={"weight": 0.7, "fillOpacity": 0.1}, hover_style={"weight": 2.5, "fillOpacity": 0.35},
        style_callback=lambda f: {"color": TIER_COLORS[f["properties"]["tier"]], "fillColor": TIER_COLORS[f["properties"]["tier"]]},
    )

    # Frame on the counties, ignoring shapes that straddle the antimeridian (Aleutians).
    b = counties.geometry.bounds
    minx, miny, maxx, maxy = counties[(b["maxx"] - b["minx"]) < 180].total_bounds
    m = Map(
        basemap=basemaps.OpenStreetMap.Mapnik,
        center=((miny + maxy) / 2, (minx + maxx) / 2),
        zoom=zoom_for_bounds(minx, miny, maxx, maxy),
        scroll_wheel_zoom=True,
        layout=widgets.Layout(height=height),
    )
    for geo in (area_layer, county_layer, zcta_layer):  # ZCTAs on top, so they take the hover when visible
        m.add(geo)
    m.add(LayersControl(position="topright", collapsed=False))
    m.add(WidgetControl(widget=info, position="bottomleft"))
    m.add(LegendControl({TIER_LABELS[t]: TIER_COLORS[t] for t in TIER_COLORS}, title="ZCTA placement", position="bottomright"))
    m.add(FullScreenControl())
    return m


state_dd = widgets.Dropdown(options=sorted(rating_areas["state"].unique()), value="MD", description="State")
map_out = widgets.Output()


def show_state(change=None):
    with map_out:
        map_out.clear_output(wait=True)
        display(build_state_map(state_dd.value))


state_dd.observe(show_state, names="value")
display(state_dd, map_out)
show_state()